# 실제 영상 연기 탐지 — YOLOv8n-cls 추론

`assets/capture1.avi` 영상에 fine-tuned 모델을 적용하여 실시간 연기 탐지 결과를 확인한다.

| 항목 | 내용 |
|---|---|
| 입력 영상 | `assets/capture1.avi` (640×480, 30fps, 약 4분 30초) |
| 모델 | `runs/smoke_detector/yolov8n_cls/weights/best.pt` |
| 출력 | 프레임별 smoke/no_smoke 라벨 + confidence 오버레이 영상 |

## 0. 경로 및 모델 로드

In [ ]:
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# ── 경로 설정 ──────────────────────────────────────────────
BASE_DIR   = Path("..").resolve()
VIDEO_PATH = BASE_DIR / "assets" / "capture1.avi"
MODEL_PATH = BASE_DIR / "runs" / "smoke_detector" / "yolov8n_cls" / "weights" / "best.pt"
OUT_DIR    = BASE_DIR / "runs" / "smoke_detector" / "video_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_VIDEO  = OUT_DIR / "capture1_result.avi"

# ── CLAHE 설정 (학습 데이터와 동일한 전처리) ──────────────
# prepare_dataset.py와 완전히 동일한 파라미터를 사용해야
# 학습-추론 간 일관성이 유지된다.
CLAHE_PROC = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def apply_clahe(img_bgr: np.ndarray) -> np.ndarray:
    """LAB L채널에만 CLAHE 적용 → 색감 유지, 밝기 대비만 정규화"""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    lab_eq = cv2.merge([CLAHE_PROC.apply(l), a, b])
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

# ── 영상 정보 확인 ────────────────────────────────────────
cap = cv2.VideoCapture(str(VIDEO_PATH))
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS          = cap.get(cv2.CAP_PROP_FPS)
W            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"입력 영상  : {VIDEO_PATH.name}")
print(f"해상도     : {W} × {H}  /  FPS: {FPS:.2f}  /  총 {TOTAL_FRAMES}프레임 ({TOTAL_FRAMES/FPS:.1f}초)")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"best.pt 없음: {MODEL_PATH}\n"
        "train_yolo.ipynb 를 먼저 실행하여 학습을 완료하세요."
    )
model = YOLO(str(MODEL_PATH))
print(f"모델 로드  : {MODEL_PATH}")
print(f"전처리     : CLAHE (clipLimit=2.0, tile=8×8) — 학습 데이터와 동일")

## 1. 영상 전체 추론 → 결과 영상 저장

모든 프레임에 연기 탐지를 수행하고 라벨을 오버레이한 영상을 저장한다.
- **초록색 박스**: no_smoke (연기 없음)
- **빨간색 박스**: smoke (연기 감지)
- 우측 상단: 현재 프레임 / 전체 프레임, 경과 시간, 처리 속도

In [ ]:
def draw_overlay(frame, label, conf, frame_idx, total, elapsed):
    h, w = frame.shape[:2]
    is_smoke  = (label == "smoke")
    bar_color = (0, 0, 200) if is_smoke else (0, 180, 0)
    cv2.rectangle(frame, (0, 0), (w, 52), bar_color, -1)
    status = f"{'[SMOKE DETECTED]' if is_smoke else '[NO SMOKE]'}  conf: {conf:.2f}"
    cv2.putText(frame, status, (10, 36),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    info = f"Frame {frame_idx}/{total}  |  {elapsed:.1f}s"
    cv2.putText(frame, info, (w - 280, 36),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
    return frame


cap    = cv2.VideoCapture(str(VIDEO_PATH))
writer = cv2.VideoWriter(str(OUT_VIDEO), cv2.VideoWriter_fourcc(*"XVID"), FPS, (W, H))

smoke_frames = 0
ms_list      = []
frame_idx    = 0
log_interval = 500
start_wall   = time.perf_counter()

print("추론 시작... (CLAHE 전처리 → 모델 추론 순서로 처리)")
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    # ① CLAHE 전처리 (학습 데이터와 동일한 정규화)
    frame_clahe = apply_clahe(frame)

    # ② YOLO 추론
    t0      = time.perf_counter()
    results = model.predict(frame_clahe, verbose=False, imgsz=224)
    ms_list.append((time.perf_counter() - t0) * 1000)

    names    = results[0].names
    top1_idx = int(results[0].probs.top1)
    label    = names[top1_idx]
    conf     = float(results[0].probs.top1conf)

    if label == "smoke":
        smoke_frames += 1

    # ③ 오버레이는 원본 프레임에 (CLAHE 적용 전 영상 저장)
    elapsed = time.perf_counter() - start_wall
    out_frame = draw_overlay(frame.copy(), label, conf, frame_idx, TOTAL_FRAMES, elapsed)
    writer.write(out_frame)

    if frame_idx % log_interval == 0 or frame_idx == TOTAL_FRAMES:
        pct    = frame_idx / TOTAL_FRAMES * 100
        avg_ms = np.mean(ms_list[-log_interval:])
        print(f"  [{pct:5.1f}%] {frame_idx}/{TOTAL_FRAMES}  |  avg {avg_ms:.1f} ms/frame  |  경과 {elapsed:.0f}s")

cap.release()
writer.release()

total_elapsed = time.perf_counter() - start_wall
smoke_ratio   = smoke_frames / max(frame_idx, 1) * 100

print("\n" + "=" * 55)
print(f"  처리 프레임   : {frame_idx}장")
print(f"  총 소요 시간  : {total_elapsed:.1f}초")
print(f"  평균 속도     : {np.mean(ms_list):.1f} ms/frame  ({1000/np.mean(ms_list):.1f} FPS)")
print(f"  smoke 프레임  : {smoke_frames}장 ({smoke_ratio:.1f}%)")
print(f"  no_smoke      : {frame_idx - smoke_frames}장 ({100-smoke_ratio:.1f}%)")
print(f"  저장 경로     : {OUT_VIDEO}")
print("=" * 55)

## 2. 시간대별 smoke 탐지 타임라인

영상 전체에서 어느 구간에 연기가 탐지됐는지 시각화한다.

In [ ]:
# 타임라인용 데이터 재수집 (추론 결과에서 smoke 여부를 초 단위로 집계)
cap      = cv2.VideoCapture(str(VIDEO_PATH))
duration = TOTAL_FRAMES / FPS           # 영상 총 길이(초)
bin_sec  = 5                            # 5초 단위로 묶어서 smoke 비율 계산
n_bins   = int(np.ceil(duration / bin_sec))
smoke_per_bin = np.zeros(n_bins)
total_per_bin = np.zeros(n_bins)

fidx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    results  = model.predict(frame, verbose=False, imgsz=224)
    names    = results[0].names
    label    = names[int(results[0].probs.top1)]
    sec      = fidx / FPS
    bin_idx  = min(int(sec // bin_sec), n_bins - 1)
    total_per_bin[bin_idx] += 1
    if label == "smoke":
        smoke_per_bin[bin_idx] += 1
    fidx += 1

cap.release()

smoke_ratio_per_bin = smoke_per_bin / np.maximum(total_per_bin, 1)
x_labels = [f"{i*bin_sec//60}:{i*bin_sec%60:02d}" for i in range(n_bins)]

# 시각화
fig, ax = plt.subplots(figsize=(16, 4))
colors  = ["tomato" if r >= 0.5 else "mediumseagreen" for r in smoke_ratio_per_bin]
bars    = ax.bar(range(n_bins), smoke_ratio_per_bin, color=colors, width=0.85)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="50% 기준선")
ax.set_xticks(range(0, n_bins, max(1, n_bins // 20)))
ax.set_xticklabels(x_labels[::max(1, n_bins // 20)], rotation=45, ha="right")
ax.set_xlabel("시간 (분:초)")
ax.set_ylabel("Smoke 비율")
ax.set_title(f"구간별 Smoke 탐지 비율  (빨강=smoke 우세, 초록=no_smoke 우세, 5초 단위)")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig(str(OUT_DIR / "smoke_timeline.png"), dpi=150)
plt.show()
print("타임라인 저장 완료")

## 3. 대표 샘플 프레임 확인

smoke / no_smoke 구간에서 각각 4장씩 뽑아 노트북 안에서 직접 확인한다.

In [ ]:
import random

SAMPLE_N = 4   # 클래스당 샘플 수

# smoke/no_smoke 프레임 인덱스 수집 (타임라인 bin 기준)
smoke_bins    = [i for i, r in enumerate(smoke_ratio_per_bin) if r >= 0.5]
no_smoke_bins = [i for i, r in enumerate(smoke_ratio_per_bin) if r < 0.5]

def pick_frame_indices(bins, n, fps, bin_sec):
    """bin 목록에서 n개를 랜덤 선택해 대표 프레임 인덱스 반환"""
    chosen = random.sample(bins, min(n, len(bins)))
    # 각 bin의 중간 프레임을 선택
    return [int((b * bin_sec + bin_sec / 2) * fps) for b in sorted(chosen)]

random.seed(0)
smoke_fidxs    = pick_frame_indices(smoke_bins,    SAMPLE_N, FPS, bin_sec)
no_smoke_fidxs = pick_frame_indices(no_smoke_bins, SAMPLE_N, FPS, bin_sec)

def grab_frame(video_path, fidx):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
    ret, frame = cap.read()
    cap.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else None

# 시각화
fig, axes = plt.subplots(2, SAMPLE_N, figsize=(SAMPLE_N * 4, 6))
for col, fidx in enumerate(smoke_fidxs):
    frame = grab_frame(VIDEO_PATH, fidx)
    res   = model.predict(frame, verbose=False, imgsz=224)
    conf  = float(res[0].probs.top1conf)
    axes[0, col].imshow(frame)
    axes[0, col].set_title(f"[SMOKE] conf={conf:.2f}\n{fidx/FPS:.1f}s", color="red")
    axes[0, col].axis("off")

for col, fidx in enumerate(no_smoke_fidxs):
    frame = grab_frame(VIDEO_PATH, fidx)
    res   = model.predict(frame, verbose=False, imgsz=224)
    conf  = float(res[0].probs.top1conf)
    axes[1, col].imshow(frame)
    axes[1, col].set_title(f"[NO SMOKE] conf={conf:.2f}\n{fidx/FPS:.1f}s", color="green")
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Smoke", fontsize=13, color="red")
axes[1, 0].set_ylabel("No Smoke", fontsize=13, color="green")
plt.suptitle("대표 샘플 프레임", fontsize=14)
plt.tight_layout()
plt.savefig(str(OUT_DIR / "sample_frames.png"), dpi=150)
plt.show()
print("샘플 프레임 저장 완료")